In [ ]:
"""
Create a Forte pipeline.
"""
import forte
import os
import yaml

nlp = forte.pipeline.Pipeline()
# An example configuration for the pipeline is provided in this repo
config_file = os.path.join('examples/pipeline/inference/config.yml')
config = yaml.safe_load(open(config_file, "r"))
config = forte.common.configuration.Config(config, default_hparams=None)
print(config)

In [ ]:
"""
Attach a reader to the pipeline.
"""
from forte.data.readers import MultiPackTerminalReader
nlp.set_reader(MultiPackTerminalReader(), config=config.reader).initialize()
# process_dataset() returns a Python iterator of data (MultiPack in this case)
# returned by `MultiPackTerminalReader`
print(next(nlp.process_dataset()).get_pack_at(0).text)

In [ ]:
"""
Add NLP processors to extract necessary language features such as
POS (Parts of Speech), Lemma and SRL (Semantic Role Labeling).
"""
from fortex.nltk.nltk_processors import NLTKLemmatizer, \
    NLTKSentenceSegmenter, NLTKWordTokenizer, NLTKPOSTagger
from fortex.allennlp.allennlp_processors import AllenNLPProcessor
from forte.data.selector import NameMatchSelector

selector = NameMatchSelector(select_name=config.reader.pack_name)
nlp.add(NLTKSentenceSegmenter(), selector=selector)
nlp.add(NLTKWordTokenizer(), selector=selector)
nlp.add(NLTKPOSTagger(), selector=selector)
nlp.add(NLTKLemmatizer(), selector=selector)
nlp.add(AllenNLPProcessor(), config=config.allennlp_query, selector=selector)

# See what's in the result data pack
nlp.initialize()
from ft.onto.base_ontology import Token, Sentence, PredicateLink
data_pack = next(nlp.process_dataset()).get_pack_at(0)
for sent in data_pack.get(Sentence):
    print("Tokens created by NLTK:")
    for token in data_pack.get(Token, sent, components=["fortex.nltk.nltk_processors.NLTKWordTokenizer"]):
        print(f"    text: {data_pack.text[token.begin:token.end]}, pos: {token.pos}, lemma: {token.lemma}")
    print("Semantic role labels created by AllenNLP:")
    for pred in data_pack.get(PredicateLink, sent, components=["fortex.allennlp.allennlp_processors.AllenNLPProcessor"]):
        verb = pred.get_parent()
        noun = pred.get_child()
        print(f"    verb: {data_pack.text[verb.begin:verb.end]}, noun: {data_pack.text[noun.begin:noun.end]}, noun_type: {pred.arg_type}")

In [ ]:
"""
Use above extracted language features to create an Elastic Search (ES) query
and retrieve relevant documents from an ES database.
"""
from typing import Any, Dict, Tuple
from forte.data.data_pack import DataPack
from forte.data.multi_pack import MultiPack
from forte.processors.base import QueryProcessor
from fortex.elastic.elastic_search_processor import ElasticSearchProcessor
# from composable_source.processors.elasticsearch_query_creator import ElasticSearchQueryCreator
from composable_source.utils.utils import query_preprocess


class ElasticSearchQueryCreator(QueryProcessor):
    """
    Complete implementation is available at composable_source/processors/elasticsearch_query_creator.py
    """

    @classmethod
    def default_configs(cls) -> Dict[str, Any]:
        return {
            "size": 1000,
            "field": "content",
            "query_pack_name": "query"
        }

    def _process_query(self, input_pack: MultiPack) -> Tuple[DataPack, Dict[str, Any]]:
        """
        process query datapack and return query
        :param input_pack:
        :return:
        """
        query_pack = input_pack.get_pack(self.configs.query_pack_name)
        query_pack.pack_name = self.configs.query_pack_name
        query = self._build_query_nlp(query_pack)
        return query_pack, query
    
    def _build_query_nlp(self, input_pack: DataPack) -> Dict[str, Any]:
        query, arg0, arg1, verb, _, is_answer_arg0 = query_preprocess(input_pack)
        if not arg0 or not arg1:
            processed_query = query
        if is_answer_arg0 is None:
            processed_query = f'{arg0} {verb} {arg1}'.lower()
        elif is_answer_arg0:
            processed_query = f'{arg1} {verb}'.lower()
        else:
            processed_query = f'{arg0} {verb}'.lower()
        return {
            "query": {
                "match_phrase": {
                    self.configs.field: {
                        "query": processed_query,
                        "slop": 10  # how far we allow the terms to be
                    }
                }
            },
            "size": self.configs.size
        }

nlp.add(ElasticSearchQueryCreator(), config=config.query_creator)
nlp.add(ElasticSearchProcessor(), config=config.indexer)

# See what's in the result data pack
nlp.initialize()
data_pack = next(nlp.process_dataset()).get_pack_at(1)
from ft.onto.base_ontology import Document
data_pack.get_single(Document)

In [ ]:
"""
Add NLP processors to extract necessary language features such as Entity links and SRL.
"""
from fortex.spacy.spacy_processors import SpacyProcessor
from forte.data.selector import RegexNameMatchSelector
pattern = rf"{config.indexer.response_pack_name_prefix}_\d"
selector = RegexNameMatchSelector(select_name=pattern)
nlp.add(component=SpacyProcessor(), config=config.spacy1, selector=selector)
nlp.add(component=SpacyProcessor(), config=config.spacy2, selector=selector)
nlp.add(AllenNLPProcessor(), config=config.allennlp, selector=selector)
nlp.add(NLTKPOSTagger(), selector=selector)
nlp.add(NLTKLemmatizer(), selector=selector)

# See what's in the result data pack
from ftx.onto.clinical import MedicalEntityMention
from ft.onto.base_ontology import Sentence
nlp.initialize()
data_pack = next(nlp.process_dataset()).get_pack_at(1)
sent = data_pack.get_single(Sentence)

print(f"Sentence: {sent.text}")
print("Entities created by SciSpacy:")
for entity in data_pack.get(MedicalEntityMention, sent):
    for umls in entity.umls_entities:
        print(f"    entity: {umls.name}, cui: {umls.cui}")
print("Semantic role labels created by AllenNLP:")
for pred in data_pack.get(PredicateLink, sent):
    verb = pred.get_parent()
    noun = pred.get_child()
    print(f"    verb: {data_pack.text[verb.begin:verb.end]}, noun: {data_pack.text[noun.begin:noun.end]}, noun_type: {pred.arg_type}")

In [ ]:
"""
Use a response creator to format the result data packs into human readable formats.
"""
from composable_source.processors.response_creator import ResponseCreator
nlp.add(ResponseCreator(), config=config.response)

# Execute the pipeline once on one query and see the results 
nlp.initialize()
data_pack = next(nlp.process_dataset())